In [1]:
from google.colab import drive
import os
import shutil
from collections import defaultdict
import csv
import re

# Install tabulate for pretty table formatting
import subprocess
subprocess.run(['pip', 'install', 'tabulate', '-q'], check=False)

from tabulate import tabulate

# Mount Google Drive
drive.mount('/content/drive')
print("✅ Google Drive mounted!")

Mounted at /content/drive
✅ Google Drive mounted!


In [2]:
# Define paths
yolo_train_path = "/content/drive/MyDrive/Railroad-crossing-Vision-Language-Model/Labeled-Data-NN/Journal-version-data/YOLO-ready-intensity-dataset-80-20-split/images/train"
yolo_test_path = "/content/drive/MyDrive/Railroad-crossing-Vision-Language-Model/Labeled-Data-NN/Journal-version-data/YOLO-ready-intensity-dataset-80-20-split/images/test"

cnn_train_path = "/content/drive/MyDrive/Railroad-crossing-Vision-Language-Model/Labeled-Data-NN/Journal-version-data/CNN-vehicle-intensity-labeled-dataset-80-20-split/train"
cnn_test_path = "/content/drive/MyDrive/Railroad-crossing-Vision-Language-Model/Labeled-Data-NN/Journal-version-data/CNN-vehicle-intensity-labeled-dataset-80-20-split/test"

# Create temporary directories to consolidate CNN images
consolidated_cnn_train = "/content/consolidated_cnn_train"
consolidated_cnn_test = "/content/consolidated_cnn_test"

# Remove if exists
if os.path.exists(consolidated_cnn_train):
    shutil.rmtree(consolidated_cnn_train)
if os.path.exists(consolidated_cnn_test):
    shutil.rmtree(consolidated_cnn_test)

# Create directories
os.makedirs(consolidated_cnn_train, exist_ok=True)
os.makedirs(consolidated_cnn_test, exist_ok=True)

print("\n" + "="*80)
print("STEP 1: CONSOLIDATING CNN IMAGES FROM SUBFOLDERS")
print("="*80)

# ==================== CONSOLIDATE TRAIN ====================
print("\n📂 Consolidating CNN TRAIN images from all classes...")
train_count = 0
train_classes_map = {}

for class_name in os.listdir(cnn_train_path):
    class_path = os.path.join(cnn_train_path, class_name)
    if not os.path.isdir(class_path):
        continue

    print(f"  • Processing class: {class_name}")

    for filename in os.listdir(class_path):
        if filename.lower().endswith(('.jpg', '.jpeg', '.png')):
            src_file = os.path.join(class_path, filename)
            dst_file = os.path.join(consolidated_cnn_train, filename)

            # Copy file
            shutil.copy2(src_file, dst_file)
            train_count += 1
            train_classes_map[filename] = class_name

print(f"\n✅ Total CNN TRAIN images consolidated: {train_count}")

# ==================== CONSOLIDATE TEST ====================
print("\n📂 Consolidating CNN TEST images from all classes...")
test_count = 0
test_classes_map = {}

for class_name in os.listdir(cnn_test_path):
    class_path = os.path.join(cnn_test_path, class_name)
    if not os.path.isdir(class_path):
        continue

    print(f"  • Processing class: {class_name}")

    for filename in os.listdir(class_path):
        if filename.lower().endswith(('.jpg', '.jpeg', '.png')):
            src_file = os.path.join(class_path, filename)
            dst_file = os.path.join(consolidated_cnn_test, filename)

            # Copy file
            shutil.copy2(src_file, dst_file)
            test_count += 1
            test_classes_map[filename] = class_name

print(f"\n✅ Total CNN TEST images consolidated: {test_count}")

# ==================== GET IMAGE LISTS ====================
print("\n" + "="*80)
print("STEP 2: MATCHING IMAGES (REMOVING _annotated SUFFIX)")
print("="*80)

def get_images_sorted(directory):
    """Get all image filenames, sorted by name"""
    images = []
    for filename in os.listdir(directory):
        if filename.lower().endswith(('.jpg', '.jpeg', '.png')):
            images.append(filename)
    return sorted(images)

def remove_annotated_suffix(filename):
    """Remove _annotated suffix from filename"""
    # Pattern: filename_annotated.ext -> filename.ext
    name_without_ext = os.path.splitext(filename)[0]
    ext = os.path.splitext(filename)[1]

    if name_without_ext.endswith('_annotated'):
        base_name = name_without_ext[:-len('_annotated')]
        return base_name + ext
    return filename

yolo_train = get_images_sorted(yolo_train_path)
cnn_train_raw = get_images_sorted(consolidated_cnn_train)
cnn_train_cleaned = [remove_annotated_suffix(f) for f in cnn_train_raw]

yolo_test = get_images_sorted(yolo_test_path)
cnn_test_raw = get_images_sorted(consolidated_cnn_test)
cnn_test_cleaned = [remove_annotated_suffix(f) for f in cnn_test_raw]

# ==================== COMPARE TRAIN ====================
print("\n" + "█"*80)
print("█ TRAIN SET COMPARISON")
print("█"*80)

yolo_train_set = set(yolo_train)
cnn_train_set = set(cnn_train_cleaned)

exact_match_train = yolo_train_set & cnn_train_set
only_yolo_train = yolo_train_set - cnn_train_set
only_cnn_train = cnn_train_set - yolo_train_set

print(f"\n📊 TRAIN COUNTS:")
print(f"  YOLO: {len(yolo_train)} images")
print(f"  CNN:  {len(cnn_train_raw)} images (raw)")
print(f"  CNN:  {len(cnn_train_cleaned)} images (after removing _annotated)")

print(f"\n📈 MATCHING ANALYSIS:")
print(f"  ✓ Exact matches: {len(exact_match_train)}")
print(f"  ✗ Only in YOLO: {len(only_yolo_train)}")
print(f"  ✗ Only in CNN:  {len(only_cnn_train)}")

if len(yolo_train_set) == len(cnn_train_set) == len(exact_match_train):
    print(f"\n✅ PERFECT MATCH - All {len(yolo_train)} images match after removing _annotated!")
else:
    print(f"\n⚠️  Partial match detected")

# ==================== TABLE: TRAIN IMAGES ====================
print(f"\n" + "="*80)
print("TABLE: TRAIN SET IMAGE NAMES COMPARISON")
print("="*80)

# Create mapping for cleaned CNN names back to raw names
cnn_train_raw_map = {remove_annotated_suffix(f): f for f in cnn_train_raw}

# Create table data for train
train_table_data = []
max_train = max(len(yolo_train), len(cnn_train_cleaned))

for i in range(max_train):
    yolo_name = yolo_train[i] if i < len(yolo_train) else "---"
    cnn_cleaned = cnn_train_cleaned[i] if i < len(cnn_train_cleaned) else "---"
    cnn_raw = cnn_train_raw_map.get(cnn_cleaned, "---") if cnn_cleaned != "---" else "---"

    match_status = "✓" if yolo_name == cnn_cleaned else "✗"
    train_table_data.append([i+1, yolo_name, cnn_raw, match_status])

# Print train table
print("\n📋 TRAIN SET - All Images (with _annotated suffix shown):")
print(tabulate(train_table_data,
               headers=["#", "YOLO Train", "CNN Train (original)", "Match"],
               tablefmt="grid",
               maxcolwidths=[5, 45, 50, 8]))

# ==================== COMPARE TEST ====================
print("\n" + "█"*80)
print("█ TEST SET COMPARISON")
print("█"*80)

yolo_test_set = set(yolo_test)
cnn_test_set = set(cnn_test_cleaned)

exact_match_test = yolo_test_set & cnn_test_set
only_yolo_test = yolo_test_set - cnn_test_set
only_cnn_test = cnn_test_set - yolo_test_set

print(f"\n📊 TEST COUNTS:")
print(f"  YOLO: {len(yolo_test)} images")
print(f"  CNN:  {len(cnn_test_raw)} images (raw)")
print(f"  CNN:  {len(cnn_test_cleaned)} images (after removing _annotated)")

print(f"\n📈 MATCHING ANALYSIS:")
print(f"  ✓ Exact matches: {len(exact_match_test)}")
print(f"  ✗ Only in YOLO: {len(only_yolo_test)}")
print(f"  ✗ Only in CNN:  {len(only_cnn_test)}")

if len(yolo_test_set) == len(cnn_test_set) == len(exact_match_test):
    print(f"\n✅ PERFECT MATCH - All {len(yolo_test)} images match after removing _annotated!")
else:
    print(f"\n⚠️  Partial match detected")

# ==================== TABLE: TEST IMAGES ====================
print(f"\n" + "="*80)
print("TABLE: TEST SET IMAGE NAMES COMPARISON")
print("="*80)

# Create mapping for cleaned CNN names back to raw names
cnn_test_raw_map = {remove_annotated_suffix(f): f for f in cnn_test_raw}

# Create table data for test
test_table_data = []
max_test = max(len(yolo_test), len(cnn_test_cleaned))

for i in range(max_test):
    yolo_name = yolo_test[i] if i < len(yolo_test) else "---"
    cnn_cleaned = cnn_test_cleaned[i] if i < len(cnn_test_cleaned) else "---"
    cnn_raw = cnn_test_raw_map.get(cnn_cleaned, "---") if cnn_cleaned != "---" else "---"

    match_status = "✓" if yolo_name == cnn_cleaned else "✗"
    test_table_data.append([i+1, yolo_name, cnn_raw, match_status])

# Print test table
print("\n📋 TEST SET - All Images (with _annotated suffix shown):")
print(tabulate(test_table_data,
               headers=["#", "YOLO Test", "CNN Test (original)", "Match"],
               tablefmt="grid",
               maxcolwidths=[5, 45, 50, 8]))

# ==================== EXPORT TABLES TO CSV ====================
print(f"\n" + "="*80)
print("EXPORTING TABLES TO CSV FILES")
print("="*80)

# Export train table to CSV
train_csv_path = "/content/train_comparison_annotated.csv"
try:
    with open(train_csv_path, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(["#", "YOLO Train", "CNN Train (original)", "Match"])
        writer.writerows(train_table_data)
    print(f"\n✅ Train comparison saved to: {train_csv_path}")
except Exception as e:
    print(f"❌ Error saving train CSV: {e}")

# Export test table to CSV
test_csv_path = "/content/test_comparison_annotated.csv"
try:
    with open(test_csv_path, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(["#", "YOLO Test", "CNN Test (original)", "Match"])
        writer.writerows(test_table_data)
    print(f"✅ Test comparison saved to: {test_csv_path}")
except Exception as e:
    print(f"❌ Error saving test CSV: {e}")

# ==================== FINAL SUMMARY ====================
print("\n" + "="*80)
print("FINAL SUMMARY")
print("="*80)

train_perfect = (len(yolo_train_set) == len(cnn_train_set) == len(exact_match_train))
test_perfect = (len(yolo_test_set) == len(cnn_test_set) == len(exact_match_test))

print(f"\n🔍 TRAIN SET (after removing _annotated suffix):")
print(f"   Status: {'✅ PERFECT MATCH' if train_perfect else '❌ MISMATCH'}")
print(f"   • Exact matches: {len(exact_match_train)}")
print(f"   • YOLO only: {len(only_yolo_train)}")
print(f"   • CNN only: {len(only_cnn_train)}")

print(f"\n🔍 TEST SET (after removing _annotated suffix):")
print(f"   Status: {'✅ PERFECT MATCH' if test_perfect else '❌ MISMATCH'}")
print(f"   • Exact matches: {len(exact_match_test)}")
print(f"   • YOLO only: {len(only_yolo_test)}")
print(f"   • CNN only: {len(only_cnn_test)}")

if train_perfect and test_perfect:
    print("\n" + "🎯 "*40)
    print("✅ SUCCESS! BOTH DATASETS MATCH PERFECTLY!")
    print("━"*80)
    print("The CNN dataset has '_annotated' suffix on all filenames.")
    print("After removing this suffix, the images match YOLO dataset 100%!")
    print("✓ SAFE TO COMPARE MODELS - Same images, different naming")
    print("🎯 "*40)
else:
    print("\n⚠️  DATASET MISMATCH AFTER REMOVING _annotated")
    print("   Check the CSV files for detailed comparison")

print("="*80)
print(f"\n📁 CSV files saved at:")
print(f"   • {train_csv_path}")
print(f"   • {test_csv_path}")
print("="*80)


STEP 1: CONSOLIDATING CNN IMAGES FROM SUBFOLDERS

📂 Consolidating CNN TRAIN images from all classes...
  • Processing class: three vehicles or more vehicles
  • Processing class: two vehicles
  • Processing class: one vehicle

✅ Total CNN TRAIN images consolidated: 332

📂 Consolidating CNN TEST images from all classes...
  • Processing class: one vehicle
  • Processing class: three vehicles or more vehicles
  • Processing class: two vehicles

✅ Total CNN TEST images consolidated: 83

STEP 2: MATCHING IMAGES (REMOVING _annotated SUFFIX)

████████████████████████████████████████████████████████████████████████████████
█ TRAIN SET COMPARISON
████████████████████████████████████████████████████████████████████████████████

📊 TRAIN COUNTS:
  YOLO: 332 images
  CNN:  332 images (raw)
  CNN:  332 images (after removing _annotated)

📈 MATCHING ANALYSIS:
  ✓ Exact matches: 332
  ✗ Only in YOLO: 0
  ✗ Only in CNN:  0

✅ PERFECT MATCH - All 332 images match after removing _annotated!

TABLE: TRA

## Check if YOLO, CNN, and VLMS are tested on the same images?

In [3]:
import os
import shutil
import csv
from tabulate import tabulate

# Define paths
yolo_test_path = "/content/drive/MyDrive/Railroad-crossing-Vision-Language-Model/Labeled-Data-NN/Journal-version-data/YOLO-ready-intensity-dataset-80-20-split/images/test"
cnn_test_path = "/content/drive/MyDrive/Railroad-crossing-Vision-Language-Model/Labeled-Data-NN/Journal-version-data/CNN-vehicle-intensity-labeled-dataset-80-20-split/test"
vlm_test_path = "/content/drive/MyDrive/Railroad-crossing-Vision-Language-Model/Labeled-Data-NN/Journal-version-data/Test-VLM-Vehicle-Count"

# Create temporary directories to consolidate images
consolidated_cnn_test = "/content/consolidated_cnn_test_3way"
consolidated_vlm_test = "/content/consolidated_vlm_test_3way"

# Remove if exists
if os.path.exists(consolidated_cnn_test):
    shutil.rmtree(consolidated_cnn_test)
if os.path.exists(consolidated_vlm_test):
    shutil.rmtree(consolidated_vlm_test)

# Create directories
os.makedirs(consolidated_cnn_test, exist_ok=True)
os.makedirs(consolidated_vlm_test, exist_ok=True)

print("\n" + "="*100)
print("THREE-WAY TEST SET COMPARISON: CNN vs YOLO vs VLMs")
print("="*100)

# ==================== CONSOLIDATE CNN TEST ====================
print("\n📂 Consolidating CNN TEST images from all class subfolders...")
cnn_test_count = 0
cnn_test_classes_map = {}

for class_name in os.listdir(cnn_test_path):
    class_path = os.path.join(cnn_test_path, class_name)
    if not os.path.isdir(class_path):
        continue

    print(f"  • Processing CNN class: {class_name}")

    for filename in os.listdir(class_path):
        if filename.lower().endswith(('.jpg', '.jpeg', '.png')):
            src_file = os.path.join(class_path, filename)
            dst_file = os.path.join(consolidated_cnn_test, filename)

            # Copy file
            shutil.copy2(src_file, dst_file)
            cnn_test_count += 1
            cnn_test_classes_map[filename] = class_name

print(f"✅ Total CNN TEST images consolidated: {cnn_test_count}")

# ==================== CONSOLIDATE VLM TEST ====================
print("\n📂 Consolidating VLM TEST images from all class subfolders...")
vlm_test_count = 0
vlm_test_classes_map = {}

for class_name in os.listdir(vlm_test_path):
    class_path = os.path.join(vlm_test_path, class_name)
    if not os.path.isdir(class_path):
        continue

    print(f"  • Processing VLM class: {class_name}")

    for filename in os.listdir(class_path):
        if filename.lower().endswith(('.jpg', '.jpeg', '.png')):
            src_file = os.path.join(class_path, filename)
            dst_file = os.path.join(consolidated_vlm_test, filename)

            # Copy file
            shutil.copy2(src_file, dst_file)
            vlm_test_count += 1
            vlm_test_classes_map[filename] = class_name

print(f"✅ Total VLM TEST images consolidated: {vlm_test_count}")

# ==================== GET IMAGE LISTS ====================
print("\n" + "="*100)
print("STEP 2: PROCESSING AND MATCHING IMAGES")
print("="*100)

def get_images_sorted(directory):
    """Get all image filenames, sorted by name"""
    images = []
    for filename in os.listdir(directory):
        if filename.lower().endswith(('.jpg', '.jpeg', '.png')):
            images.append(filename)
    return sorted(images)

def remove_annotated_suffix(filename):
    """Remove _annotated suffix from filename"""
    # Pattern: filename_annotated.ext -> filename.ext
    name_without_ext = os.path.splitext(filename)[0]
    ext = os.path.splitext(filename)[1]

    if name_without_ext.endswith('_annotated'):
        base_name = name_without_ext[:-len('_annotated')]
        return base_name + ext
    return filename

# Get image lists
yolo_test = get_images_sorted(yolo_test_path)
cnn_test_raw = get_images_sorted(consolidated_cnn_test)
cnn_test_cleaned = [remove_annotated_suffix(f) for f in cnn_test_raw]
vlm_test = get_images_sorted(consolidated_vlm_test)

# Convert to sets for comparison
yolo_test_set = set(yolo_test)
cnn_test_set = set(cnn_test_cleaned)
vlm_test_set = set(vlm_test)

# ==================== THREE-WAY ANALYSIS ====================
print("\n" + "█"*100)
print("█ TEST SET COMPARISON: ALL THREE DATASETS")
print("█"*100)

# Calculate intersections and differences
all_three_match = yolo_test_set & cnn_test_set & vlm_test_set
yolo_cnn_only = (yolo_test_set & cnn_test_set) - vlm_test_set
yolo_vlm_only = (yolo_test_set & vlm_test_set) - cnn_test_set
cnn_vlm_only = (cnn_test_set & vlm_test_set) - yolo_test_set
yolo_only = yolo_test_set - cnn_test_set - vlm_test_set
cnn_only = cnn_test_set - yolo_test_set - vlm_test_set
vlm_only = vlm_test_set - yolo_test_set - cnn_test_set

print(f"\n📊 IMAGE COUNTS:")
print(f"  YOLO: {len(yolo_test)} images")
print(f"  CNN:  {len(cnn_test_raw)} images (raw with _annotated)")
print(f"  CNN:  {len(cnn_test_cleaned)} images (after removing _annotated suffix)")
print(f"  VLM:  {len(vlm_test)} images")

print(f"\n📈 MATCHING ANALYSIS:")
print(f"  ✓ In all three datasets:        {len(all_three_match)}")
print(f"  ✓ In YOLO & CNN only:           {len(yolo_cnn_only)}")
print(f"  ✓ In YOLO & VLM only:           {len(yolo_vlm_only)}")
print(f"  ✓ In CNN & VLM only:            {len(cnn_vlm_only)}")
print(f"  ✗ Only in YOLO:                 {len(yolo_only)}")
print(f"  ✗ Only in CNN:                  {len(cnn_only)}")
print(f"  ✗ Only in VLM:                  {len(vlm_only)}")

perfect_match = (len(yolo_test_set) == len(cnn_test_set) == len(vlm_test_set) == len(all_three_match))

if perfect_match:
    print(f"\n✅ PERFECT MATCH - All {len(yolo_test)} images match across all three datasets!")
else:
    print(f"\n⚠️  PARTIAL MATCH - Some images differ across datasets")

# ==================== TABLE: TEST IMAGES ====================
print(f"\n" + "="*100)
print("TABLE: TEST SET IMAGE NAMES COMPARISON (All three datasets)")
print("="*100)

# Create mappings for cleaned CNN names back to raw names
cnn_test_raw_map = {remove_annotated_suffix(f): f for f in cnn_test_raw}

# Get unique filenames from all three sets
all_unique_names = sorted(yolo_test_set | cnn_test_set | vlm_test_set)

# Create table data
test_table_data = []

for i, img_name in enumerate(all_unique_names):
    # Check if in each dataset
    in_yolo = "✓" if img_name in yolo_test_set else "✗"
    in_cnn = "✓" if img_name in cnn_test_set else "✗"
    in_vlm = "✓" if img_name in vlm_test_set else "✗"

    # Get CNN raw name if exists
    cnn_raw_name = cnn_test_raw_map.get(img_name, "---")

    # Get class information
    cnn_class = cnn_test_classes_map.get(cnn_raw_name, "---") if cnn_raw_name != "---" else "---"
    vlm_class = vlm_test_classes_map.get(img_name, "---") if in_vlm == "✓" else "---"

    # Check overall match status
    if in_yolo == "✓" and in_cnn == "✓" and in_vlm == "✓":
        match_status = "✓ All"
    elif in_yolo == "✓" and in_cnn == "✓":
        match_status = "⚠ Y+C"
    elif in_yolo == "✓" and in_vlm == "✓":
        match_status = "⚠ Y+V"
    elif in_cnn == "✓" and in_vlm == "✓":
        match_status = "⚠ C+V"
    else:
        match_status = "✗ Unique"

    test_table_data.append([
        i+1,
        img_name,
        in_yolo,
        in_cnn,
        in_vlm,
        match_status,
        cnn_class,
        vlm_class
    ])

# Print test table
print("\n📋 TEST SET - All Images Across Three Datasets:")
print(tabulate(test_table_data,
               headers=["#", "Image Name", "YOLO", "CNN", "VLM", "Status", "CNN Class", "VLM Class"],
               tablefmt="grid",
               maxcolwidths=[4, 50, 6, 6, 6, 10, 15, 15]))

# ==================== EXPORT TABLES TO CSV ====================
print(f"\n" + "="*100)
print("EXPORTING DETAILED COMPARISON TO CSV FILES")
print("="*100)

# Export main comparison table to CSV
test_csv_path = "/content/test_comparison_3way_all_datasets.csv"
try:
    with open(test_csv_path, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(["#", "Image Name", "YOLO", "CNN", "VLM", "Status", "CNN Class", "VLM Class"])
        writer.writerows(test_table_data)
    print(f"\n✅ Main comparison saved to: {test_csv_path}")
except Exception as e:
    print(f"❌ Error saving test CSV: {e}")

# Export summary statistics
summary_csv_path = "/content/test_comparison_3way_summary.csv"
try:
    with open(summary_csv_path, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(["Metric", "Count"])
        writer.writerow(["Total YOLO images", len(yolo_test)])
        writer.writerow(["Total CNN images (raw)", len(cnn_test_raw)])
        writer.writerow(["Total CNN images (cleaned)", len(cnn_test_cleaned)])
        writer.writerow(["Total VLM images", len(vlm_test)])
        writer.writerow(["", ""])
        writer.writerow(["In all three datasets", len(all_three_match)])
        writer.writerow(["In YOLO & CNN only", len(yolo_cnn_only)])
        writer.writerow(["In YOLO & VLM only", len(yolo_vlm_only)])
        writer.writerow(["In CNN & VLM only", len(cnn_vlm_only)])
        writer.writerow(["Only in YOLO", len(yolo_only)])
        writer.writerow(["Only in CNN", len(cnn_only)])
        writer.writerow(["Only in VLM", len(vlm_only)])
    print(f"✅ Summary statistics saved to: {summary_csv_path}")
except Exception as e:
    print(f"❌ Error saving summary CSV: {e}")

# Export mismatches only
mismatches_csv_path = "/content/test_comparison_3way_mismatches.csv"
try:
    with open(mismatches_csv_path, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(["Image Name", "In YOLO", "In CNN", "In VLM", "Status", "CNN Class", "VLM Class"])
        for row in test_table_data:
            if row[5] != "✓ All":  # Only mismatches
                writer.writerow(row[1:])  # Skip the index column
    print(f"✅ Mismatches only saved to: {mismatches_csv_path}")
except Exception as e:
    print(f"❌ Error saving mismatches CSV: {e}")

# ==================== FINAL SUMMARY ====================
print("\n" + "="*100)
print("FINAL SUMMARY - TEST SET ANALYSIS")
print("="*100)

print(f"\n🔍 DATASET COMPARISON RESULTS:")
print(f"   Overall Status: {'✅ PERFECT MATCH' if perfect_match else '❌ MISMATCH DETECTED'}")
print(f"\n   📊 Totals:")
print(f"      • YOLO:  {len(yolo_test):4d} images")
print(f"      • CNN:   {len(cnn_test_cleaned):4d} images (after removing _annotated)")
print(f"      • VLM:   {len(vlm_test):4d} images")

print(f"\n   🎯 Matching Summary:")
print(f"      • All three match:     {len(all_three_match):4d} images ✓")
print(f"      • YOLO & CNN only:     {len(yolo_cnn_only):4d} images")
print(f"      • YOLO & VLM only:     {len(yolo_vlm_only):4d} images")
print(f"      • CNN & VLM only:      {len(cnn_vlm_only):4d} images")
print(f"      • Unique to YOLO:      {len(yolo_only):4d} images")
print(f"      • Unique to CNN:       {len(cnn_only):4d} images")
print(f"      • Unique to VLM:       {len(vlm_only):4d} images")

if perfect_match:
    print("\n" + "🎯 "*50)
    print("✅ SUCCESS! ALL THREE DATASETS MATCH PERFECTLY!")
    print("━"*100)
    print("• CNN images have '_annotated' suffix (now removed for comparison)")
    print("• YOLO images are in a single folder (flat structure)")
    print("• VLM images are organized in class subfolders")
    print("✓ SAFE TO COMPARE ALL THREE MODELS - Same images, different organization")
    print("🎯 "*50)
else:
    print("\n⚠️  DATASET MISMATCH DETECTED")
    print("   Check the CSV files for detailed comparison:")
    print(f"   • Main comparison: {test_csv_path}")
    print(f"   • Mismatches only: {mismatches_csv_path}")

print("\n" + "="*100)
print("📁 OUTPUT CSV FILES:")
print("="*100)
print(f"1️⃣  Complete comparison (all images):")
print(f"    {test_csv_path}")
print(f"\n2️⃣  Summary statistics:")
print(f"    {summary_csv_path}")
print(f"\n3️⃣  Mismatches only (images not in all datasets):")
print(f"    {mismatches_csv_path}")
print("="*100)


THREE-WAY TEST SET COMPARISON: CNN vs YOLO vs VLMs

📂 Consolidating CNN TEST images from all class subfolders...
  • Processing CNN class: one vehicle
  • Processing CNN class: three vehicles or more vehicles
  • Processing CNN class: two vehicles
✅ Total CNN TEST images consolidated: 83

📂 Consolidating VLM TEST images from all class subfolders...
  • Processing VLM class: One Vehicle
  • Processing VLM class: Two Vehicles
  • Processing VLM class: Three Vehicles or More
  • Processing VLM class: .ipynb_checkpoints
✅ Total VLM TEST images consolidated: 83

STEP 2: PROCESSING AND MATCHING IMAGES

████████████████████████████████████████████████████████████████████████████████████████████████████
█ TEST SET COMPARISON: ALL THREE DATASETS
████████████████████████████████████████████████████████████████████████████████████████████████████

📊 IMAGE COUNTS:
  YOLO: 83 images
  CNN:  83 images (raw with _annotated)
  CNN:  83 images (after removing _annotated suffix)
  VLM:  83 images

📈 MA